# 159 — Proyecto: plataforma de IA observable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

El proyecto integra las clases 148-158 en una **plataforma de IA observable**:
capacidades compartidas que todo caso de uso reutiliza. Piezas y su clase:

- **Trazabilidad y ciclo de vida (145-146)**: todo artefacto (dataset, modelo,
  prompt, agente) con versión, linaje y semilla.
- **Registro y promoción (150)**: challenger → champion con criterios
  explícitos y baseline.
- **CI/CD con evals (148, 151)**: cada cambio pasa pruebas y evaluación
  offline antes del tráfico.
- **Serving (152) + observabilidad (153)**: telemetría OpenTelemetry con
  atributos de IA (versión de modelo/prompt, tokens, costo, id de
  trayectoria).
- **Deriva y trayectorias (151-153)**, **economía (157)**, **resiliencia
  (158)**.

Conceptos de operación: un **SLI** es la medición; un **SLO** el objetivo
(«p95 < 2 s el 99 % del mes»); el **presupuesto de error** es el complemento
del SLO y funciona como moneda: si se gasta rápido, se congelan lanzamientos.
En IA se añaden SLO de calidad (aprobación de evals online) y de costo
(USD/1k peticiones). El ciclo de release: cambio → CI + evals offline →
challenger registrado → canary con telemetría comparada → promoción o
rollback → operación. **Cada flecha produce evidencia**; sin evidencia por
etapa no hay plataforma observable, hay dashboards.


## 🧮 Ejemplo de referencia

Promoción del prompt `v12` sobre el champion `v11` (SLO: p95 < 2 s, evals
≥ 90 %, costo ≤ 12 USD/1k):

```text
evals offline (n=500):  v12 93 %  vs  v11 89 %      → pasa
canary 10 %, 48 h:      p95 1.7 s · costo 11.2 · aprobación online 91 %
decisión:               promover; v11 queda como objetivo de rollback
día 5:                  aprobación online cae a 84 %  → rollback a v11
análisis (156):         la calidad NO se recupera → la causa era deriva de
                        datos (consultas nuevas), no el prompt v12
```

La evidencia por etapa (reporte de evals, diff canary, traza del rollback) es
lo que permitió atribuir la causa en horas; sin linaje ni versiones, el
rollback habría parecido «no funcionar» sin explicación.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("capstone", seed=159)
show(result)


## Reflexión

1. En el ejemplo, el rollback no recuperó la calidad: ¿qué evidencia concreta
   distinguió «prompt malo» de «deriva de datos», y qué habrías concluido sin
   ella?
2. ¿Por qué un SLO de disponibilidad y latencia es insuficiente para un
   servicio de LLM, y qué dos SLO adicionales propone la materia con qué
   riesgo de medirlos mal?
3. Si tu presupuesto de error mensual se agota en la primera semana, ¿qué
   decisión organizacional dispara esa señal según el modelo SRE y por qué es
   preferible a «intentar tener más cuidado»?
